<a href="https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/wip-text-generation-rnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RNN-based Text Generation for Names

This notebook demonstrates how to build a character-level Recurrent Neural Network (RNN) to generate new names. We'll use PyTorch to implement a vanilla RNN model that learns patterns from a dataset of names and generates new, plausible-sounding names.

## Overview

1. **Data Preparation**: Load a dataset of names and process it for training
2. **Model Architecture**: Build an RNN with embedding layers and dropout
3. **Training**: Train the model on sequences of characters
4. **Name Generation**: Generate new names using the trained model
5. **Temperature-based Sampling**: Explore how temperature affects creativity in generation

## Data Preparation

We start by importing the necessary libraries for our task. Then we download a dataset of human names from Andrej Karpathy's GitHub repository. The dataset contains common first names which we'll use to train our RNN model.

### Data Processing Steps:
1. Download the names dataset
2. Extract all unique characters to build our vocabulary
3. Add special tokens for sequence start `<s>` and end `<e>`
4. Create character-to-index and index-to-character mappings
5. Prepare sequences for training by creating input-target pairs

For each name, we create sequences of length 5 characters (configurable) and set the following character as the target. This is how the model learns to predict the next character based on the previous 5 characters.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import random
import requests

# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Download the names dataset
url = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"
response = requests.get(url)
names_data = response.text.strip().split('\n')
print(f"Dataset loaded: {len(names_data)} names")
print(f"Sample names: {random.sample(names_data, 5)}")

### Preprocessing: Creating Character Vocabulary

Next, we need to create a vocabulary of all characters in our dataset. This vocabulary will be used to encode the characters as numerical indices that the model can process. We also add special tokens:
- `<s>`: Start token to mark the beginning of a name
- `<e>`: End token to mark the end of a name

These special tokens help the model learn when to start and end name generation.

In [ ]:
# Process the data - FIXED ORDER: First process data, then add special tokens
all_chars = set(''.join(names_data))
print(f"Characters: {sorted(list(all_chars))}")

# Add special tokens FIRST before creating char_to_idx
all_chars.add('<s>')  # Start token
all_chars.add('<e>')  # End token

# Now create the mappings
char_to_idx = {ch: i for i, ch in enumerate(sorted(all_chars))}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}
vocab_size = len(char_to_idx)

print(f"Vocabulary size: {vocab_size}")
print(f"Char to idx mapping: {char_to_idx}")

### Creating Training Sequences

For training our RNN, we need to create sequences of characters along with their corresponding target characters. We use a sliding window approach:

1. For each name, we add start and end tokens
2. We create sequences of length `seq_length` (in this case, 5)
3. For each sequence, the target is the character that follows immediately after

For example, with the name "Emma" and `seq_length=3`:
- Input: `<s><s>E` → Target: `m`
- Input: `<s>Em` → Target: `m`
- Input: `Emm` → Target: `a`
- Input: `mma` → Target: `<e>`

This way, our model learns to predict the next character based on the previous few characters.

In [ ]:
# Prepare training data with sequences instead of single characters
def create_sequence_data(names, char_to_idx, seq_length=5):
    X, y = [], []

    for name in names:
        # Add start and end tokens as whole tokens, not as characters
        chars = ['<s>'] + list(name) + ['<e>']

        # Create sequences of length seq_length
        for i in range(len(chars) - seq_length):
            sequence = chars[i:i+seq_length]
            target = chars[i+seq_length]

            # Convert to indices
            seq_idx = [char_to_idx[c] for c in sequence]
            target_idx = char_to_idx[target]

            X.append(seq_idx)
            y.append(target_idx)

    return torch.tensor(X, dtype=torch.long).to(device), torch.tensor(y, dtype=torch.long).to(device)

# Create sequence data
seq_length = 5
X, y = create_sequence_data(names_data, char_to_idx, seq_length)
print(f"Training data created: {len(X)} examples")
print(f"Sample input sequence: {X[0].tolist()} → target: {y[0].item()}")
print(f"As characters: {''.join([idx_to_char[idx] for idx in X[0].tolist()])} → {idx_to_char[y[0].item()]}")

## Model Architecture

Now we define our RNN model architecture. We're using a vanilla RNN with the following components:

1. **Embedding Layer**: Converts character indices to dense vectors of dimension `embedding_dim`
2. **RNN Layer**: Processes sequences and captures temporal patterns with `hidden_size` dimensionality
3. **Dropout**: Prevents overfitting by randomly zeroing some values during training
4. **Linear Output Layer**: Maps RNN outputs to character probabilities (vocabulary size)

The model architecture includes several improvements over a basic RNN:

- Multiple RNN layers (`num_layers=2`) for more representational capacity
- Dropout regularization (`dropout=0.2`) to prevent overfitting
- Tanh activation function for better gradient flow compared to sigmoid
- Embedding layer to represent characters as dense vectors

This design allows the model to learn complex patterns in name sequences while mitigating common issues like overfitting and vanishing gradients.

In [ ]:
# Create an improved RNN model
class ImprovedNameRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers=2, dropout=0.2):
        super(ImprovedNameRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # RNN layer as requested (keeping vanilla RNN)
        self.rnn = nn.RNN(
            embedding_dim,
            hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True,
            nonlinearity='tanh'  # Using tanh for better gradient flow
        )

        # Output layer
        self.fc = nn.Linear(hidden_size, vocab_size)

        # Dropout
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, hidden=None):
        batch_size = x.size(0)

        # Initialize hidden state if not provided
        if hidden is None:
            hidden = torch.zeros(self.num_layers, batch_size, self.hidden_size, device=x.device)

        # Embedding
        x = self.embedding(x)

        # Apply dropout to embeddings
        x = self.dropout(x)

        # RNN
        out, hidden = self.rnn(x, hidden)

        # Apply dropout
        out = self.dropout(out[:, -1, :])  # Take only the last output

        # Output layer
        out = self.fc(out)

        return out, hidden

### Model Initialization and Training Setup

With our model architecture defined, we now initialize the model with appropriate hyperparameters and set up the training components:

- **Model Parameters**:
  - `embedding_dim=64`: Size of the character embedding vectors
  - `hidden_size=256`: Number of features in the hidden state
  - `num_layers=2`: Number of recurrent layers
  - `dropout=0.2`: Dropout rate for regularization

- **Loss Function**: Cross-Entropy Loss, appropriate for classification problems

- **Optimizer**: Adam with a learning rate of 0.002 and weight decay for regularization

- **Learning Rate Scheduler**: ReduceLROnPlateau to reduce the learning rate when the validation loss plateaus

These components are essential for effective training of our RNN model.

In [ ]:
# Model parameters
embedding_dim = 64
hidden_size = 256
num_layers = 2
dropout = 0.2

# Create model
model = ImprovedNameRNN(vocab_size, embedding_dim, hidden_size, num_layers, dropout).to(device)
print(f"Model created with {sum(p.numel() for p in model.parameters())} parameters")

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.002, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2, factor=0.5)

## Training Process

With our model architecture defined, we now set up the training process. The training includes several best practices:

1. **Data Splitting**: Separate data into training and validation sets (90%/10% split)
2. **Mini-batch Training**: Process data in batches of size 512 for efficient training
3. **Training Monitoring**: Track both training and validation loss throughout training
4. **Learning Rate Scheduling**: Reduce learning rate when validation loss plateaus
5. **Early Stopping**: Save the best model based on validation performance
6. **Gradient Clipping**: Prevent exploding gradients by clipping to a max norm of 5

We train for 100 epochs with careful monitoring to achieve optimal performance.

In [ ]:
# Training parameters
batch_size = 512  # Increased batch size for faster training
num_epochs = 100   # More epochs
print_every = 500

# Training loop
train_losses = []
val_losses = []

# Split data into train and validation sets
val_ratio = 0.1
val_size = int(len(X) * val_ratio)
indices = torch.randperm(len(X))
train_indices = indices[val_size:]
val_indices = indices[:val_size]

X_train, y_train = X[train_indices], y[train_indices]
X_val, y_val = X[val_indices], y[val_indices]

print(f"Training set: {len(X_train)} examples")
print(f"Validation set: {len(X_val)} examples")

### Helper Functions for Training

We define two helper functions for the training process:

1. **get_batch**: Creates random mini-batches from the training data for each training step
2. **evaluate**: Evaluates the model on the validation dataset to monitor generalization performance

In [ ]:
def get_batch(X, y, batch_size):
    indices = torch.randperm(len(X), device=device)[:batch_size]
    return X[indices], y[indices]

def evaluate(model, X, y, batch_size):
    model.eval()
    total_loss = 0
    num_batches = len(X) // batch_size
    with torch.no_grad():
        for i in range(num_batches):
            start_idx = i * batch_size
            end_idx = start_idx + batch_size
            inputs = X[start_idx:end_idx]
            targets = y[start_idx:end_idx]

            outputs, _ = model(inputs)
            loss = criterion(outputs, targets)
            total_loss += loss.item()

    return total_loss / num_batches

### Main Training Loop

Now we implement the main training loop that will:

1. Iterate through the specified number of epochs
2. For each epoch, train on mini-batches from the training set
3. Evaluate on the validation set at the end of each epoch
4. Update the learning rate based on validation performance
5. Save the best model based on validation loss
6. Visualize the training process with loss curves

In [ ]:
# Training loop
best_val_loss = float('inf')
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    num_batches = len(X_train) // batch_size

    for i in range(num_batches):
        # Get mini-batch
        inputs, targets = get_batch(X_train, y_train, batch_size)

        # Forward pass
        outputs, _ = model(inputs)
        loss = criterion(outputs, targets)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
        optimizer.step()

        total_loss += loss.item()

        if (i+1) % print_every == 0 or i == num_batches-1:
            print(f'Epoch {epoch+1}/{num_epochs}, Batch {i+1}/{num_batches}, Loss: {loss.item():.4f}')

    avg_train_loss = total_loss / num_batches
    train_losses.append(avg_train_loss)

    # Evaluate on validation set
    val_loss = evaluate(model, X_val, y_val, batch_size)
    val_losses.append(val_loss)

    print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {val_loss:.4f}')

    # Update learning rate based on validation loss
    scheduler.step(val_loss)

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        # Save the model (optional)
        torch.save(model.state_dict(), 'best_name_rnn.pt')

    # Get current learning rate
    current_lr = optimizer.param_groups[0]['lr']
    print(f'Learning rate: {current_lr}')

# Plot the training and validation loss
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

## Name Generation

After training the model, we use it to generate new names. This involves a character-by-character generation process:

1. Start with a sequence containing the start token `<s>`
2. Feed the sequence into the model to predict the next character
3. Sample the next character from the model's predicted probability distribution
4. Add the new character to the generated name and update the input sequence
5. Repeat until we either reach the end token `<e>` or a maximum length

### Temperature Sampling

We implement temperature sampling to control the randomness of generation:

- **Lower temperature** (e.g., 0.5): Makes the distribution more peaked, leading to more predictable and common names
- **Neutral temperature** (1.0): Uses the exact probabilities learned by the model
- **Higher temperature** (e.g., 1.5): Flattens the distribution, increasing diversity and creativity

The temperature parameter divides the logits before softmax, effectively controlling how closely the model follows the learned distribution.

In [ ]:
# Function to generate a name
def generate_name(model, char_to_idx, idx_to_char, seq_length=5, max_length=20, temperature=1.0):
    model.eval()
    with torch.no_grad():
        # Start with start token and padding
        current_seq = ['<s>'] + ['<s>'] * (seq_length - 1)
        name = []

        for _ in range(max_length):
            # Convert sequence to indices
            seq_idx = [char_to_idx[c] for c in current_seq]
            seq_tensor = torch.tensor([seq_idx], dtype=torch.long, device=device)

            # Get prediction
            output, _ = model(seq_tensor)

            # Apply temperature
            if temperature != 1.0:
                output = output / temperature

            # Apply softmax to get probabilities
            probs = torch.softmax(output, dim=1)

            # Sample from the distribution
            next_char_idx = torch.multinomial(probs, 1).item()
            next_char = idx_to_char[next_char_idx]

            # If end token, stop generation
            if next_char == '<e>':
                break

            # Add to name
            name.append(next_char)

            # Update sequence
            current_seq = current_seq[1:] + [next_char]

        return ''.join(name)

### Generate New Names

Now we'll use our trained model to generate new names. We'll first try to load the best model that was saved during training, and then generate:

1. A batch of 10 names using the default temperature setting
2. Multiple batches using different temperature values to see how it affects creativity

This will demonstrate how well our model has learned to generate plausible new names that follow patterns in the training data.

In [ ]:
# Try to load the best model if it exists, otherwise use the current model
try:
    model.load_state_dict(torch.load('best_name_rnn.pt'))
    print("Loaded the best model")
except:
    print("Using the current model")

# Generate 10 names
print("\nGenerated Names:")
for _ in range(10):
    name = generate_name(model, char_to_idx, idx_to_char, seq_length)
    print(name)

# Generate names with different temperatures
print("\nNames with different temperatures:")
for temp in [0.5, 0.7, 1.0, 1.2, 1.5]:
    print(f"\nTemperature: {temp}")
    for _ in range(5):
        name = generate_name(model, char_to_idx, idx_to_char, seq_length, temperature=temp)
        print(name)

## Conclusion

In this notebook, we've successfully built and trained a character-level RNN model for generating names. We've demonstrated:

1. **Data Processing**: Preparing character sequences from a names dataset
2. **RNN Implementation**: Building a PyTorch RNN with embeddings and dropout
3. **Effective Training**: Using best practices like learning rate scheduling and early stopping
4. **Creative Generation**: Implementing temperature sampling to control creativity

The model can generate new, plausible-sounding names that follow patterns learned from the training data. We can also control the creativity of the generation by adjusting the temperature parameter.

### Future Improvements

To extend this project, you could:

- Try more advanced architectures like LSTM or Transformer models
- Add conditioning to generate names from specific cultures or languages
- Implement beam search for better generation quality
- Fine-tune hyperparameters to improve model performance
- Add a user interface for interactive name generation